# Dự đoán doanh số bán hàng - Online Retail II

**Bài toán:** Hồi quy - dự đoán giá trị mỗi hóa đơn (`InvoiceTotal`) dựa trên thời gian, quốc gia, số dòng hàng và lịch sử khách hàng.

**Lưu ý thiết kế (tránh data leakage):** `TotalSales = Quantity × Price`, nên nếu dùng `Quantity`/`Price` làm đặc trưng thì mô hình chỉ học phép nhân. Notebook gộp dữ liệu theo **hóa đơn** và không dùng `Quantity`/`Price` làm đầu vào.

**Quy trình:** Tải dữ liệu → EDA → Làm sạch → Feature engineering → Chia train/test theo thời gian → Huấn luyện nhiều mô hình → Đánh giá → Tinh chỉnh → Diễn giải.

## 0. Cấu hình & import

In [ ]:
import time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

# ---------------- CẤU HÌNH ----------------
DATA_DIR = Path("data")
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
SPLIT_DATE = "2011-10-01"      # train: trước ngày này, test: từ ngày này
OUTLIER_Q = 0.995              # loại hóa đơn có giá trị trên phân vị này (đơn sỉ cực lớn)
TOP_COUNTRIES = 10             # giữ top N quốc gia, còn lại gộp "Other"
MAX_TRAIN_KNN = 30000          # giới hạn mẫu train cho KNN
MAX_TRAIN_SVR = 15000          # giới hạn mẫu train cho SVR (SVR ~ O(n^2..n^3))
USE_CACHE = True

## 1. Tải dữ liệu (Chapter 2)

In [ ]:
def find_data_file():
    for ext in (".xlsx", ".csv"):
        p = DATA_DIR / f"online_retail_II{ext}"
        if p.exists():
            return p
    cands = list(DATA_DIR.glob("*.xlsx")) + list(DATA_DIR.glob("*.csv"))
    if cands:
        return cands[0]
    raise FileNotFoundError(
        "Không tìm thấy dataset trong thư mục data/. Hãy tải Online Retail II từ UCI "
        "và đặt vào data/online_retail_II.xlsx (hoặc .csv).")

def load_raw():
    cache = DATA_DIR / "online_retail_II.pkl"
    if USE_CACHE and cache.exists():
        print("Đọc từ cache:", cache)
        return pd.read_pickle(cache)
    path = find_data_file()
    print("Đọc file:", path)
    if path.suffix == ".csv":
        try:
            df = pd.read_csv(path, dtype={"Invoice": str, "StockCode": str})
        except UnicodeDecodeError:
            df = pd.read_csv(path, dtype={"Invoice": str, "StockCode": str}, encoding="latin-1")
    else:
        sheets = pd.read_excel(path, sheet_name=None, dtype={"Invoice": str, "StockCode": str})
        df = pd.concat(sheets.values(), ignore_index=True)
    df = df.rename(columns={"Customer ID": "CustomerID"})
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    df["Invoice"] = df["Invoice"].astype(str)
    df["StockCode"] = df["StockCode"].astype(str)
    # tối ưu bộ nhớ
    df["Quantity"] = df["Quantity"].astype("int32")
    df["Price"] = df["Price"].astype("float32")
    df["Country"] = df["Country"].astype("category")
    if USE_CACHE:
        df.to_pickle(cache)
    return df

raw = load_raw()
print(raw.shape)
raw.head()

In [ ]:
raw.info(memory_usage="deep")

## 2. Khám phá & thống kê mô tả (Chapter 4)

In [ ]:
miss = pd.DataFrame({"n_missing": raw.isna().sum(), "pct": (raw.isna().mean() * 100).round(2)})
miss

In [ ]:
raw[["Quantity", "Price"]].describe().T

In [ ]:
print("Quantity <= 0 :", (raw["Quantity"] <= 0).sum())
print("Price    <= 0 :", (raw["Price"] <= 0).sum())
print("Invoice bắt đầu bằng 'C' (hủy):", raw["Invoice"].str.startswith("C").sum())
print("Dòng trùng lặp:", raw.duplicated().sum())
print("StockCode không phải mã sản phẩm (không bắt đầu bằng 5 chữ số):",
      (~raw["StockCode"].str.match(r"^\d{5}")).sum())
print(raw.loc[~raw["StockCode"].str.match(r"^\d{5}"), "StockCode"].value_counts().head(10))
print("Khoảng thời gian:", raw["InvoiceDate"].min(), "->", raw["InvoiceDate"].max())

## 3. Làm sạch dữ liệu (Chapter 5)

- Bỏ dòng trùng lặp
- Bỏ hóa đơn hủy (`Invoice` bắt đầu bằng `C`)
- Giữ `Quantity > 0` và `Price > 0`
- Bỏ các `StockCode` phi sản phẩm (`POST`, `D`, `M`, `BANK CHARGES`... : phí ship, chiết khấu, điều chỉnh)
- `CustomerID` thiếu: **không drop**, đánh dấu là khách vãng lai (`is_guest`) để giữ dữ liệu

In [ ]:
df = raw.drop_duplicates().copy()
n0 = len(df)
df = df[~df["Invoice"].str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
df = df[df["StockCode"].str.match(r"^\d{5}")]
print(f"Số dòng: {len(raw):,} -> sau drop trùng {n0:,} -> sau làm sạch {len(df):,}")
df["TotalSales"] = df["Quantity"] * df["Price"]
df.head()

## 4. Feature Engineering - gộp theo hóa đơn

Mỗi dòng = 1 hóa đơn. Đặc trưng:
- Thời gian: `Month`, `DayOfWeek`, `Hour`, `DayOfMonth`, `IsWeekend`
- `Country` (gộp nước hiếm thành "Other")
- `n_lines`: số mặt hàng khác nhau trong hóa đơn
- Lịch sử khách hàng **chỉ từ quá khứ**: `n_prev_orders`, `avg_prev_spend`, `days_since_last`
- `is_guest`: không có `CustomerID`

Target: `InvoiceTotal` = tổng `Quantity × Price` của hóa đơn.

In [ ]:
inv = (df.groupby("Invoice")
         .agg(InvoiceDate=("InvoiceDate", "min"),
              CustomerID=("CustomerID", "first"),
              Country=("Country", "first"),
              n_lines=("StockCode", "nunique"),
              InvoiceTotal=("TotalSales", "sum"))
         .reset_index()
         .sort_values("InvoiceDate")
         .reset_index(drop=True))
inv["Country"] = inv["Country"].astype(str)

# loại outlier cực lớn
cap = inv["InvoiceTotal"].quantile(OUTLIER_Q)
print(f"Ngưỡng outlier (phân vị {OUTLIER_Q}): {cap:,.2f}  | loại {(inv.InvoiceTotal > cap).sum()} hóa đơn")
inv = inv[inv["InvoiceTotal"] <= cap].reset_index(drop=True)

# đặc trưng thời gian
d = inv["InvoiceDate"]
inv["Month"] = d.dt.month
inv["DayOfMonth"] = d.dt.day
inv["DayOfWeek"] = d.dt.dayofweek
inv["Hour"] = d.dt.hour
inv["IsWeekend"] = (inv["DayOfWeek"] >= 5).astype(int)

# gộp quốc gia hiếm
top = inv["Country"].value_counts().head(TOP_COUNTRIES).index
inv["Country"] = np.where(inv["Country"].isin(top), inv["Country"], "Other")

# đặc trưng khách hàng (chỉ dùng quá khứ, hóa đơn đã sort theo thời gian)
inv["is_guest"] = inv["CustomerID"].isna().astype(int)
g = inv.groupby("CustomerID")
inv["n_prev_orders"] = g.cumcount()
prev_sum = g["InvoiceTotal"].cumsum() - inv["InvoiceTotal"]
inv["avg_prev_spend"] = np.where(inv["n_prev_orders"] > 0, prev_sum / inv["n_prev_orders"].replace(0, np.nan), np.nan)
inv["days_since_last"] = g["InvoiceDate"].diff().dt.total_seconds() / 86400
# khách vãng lai: không có lịch sử
guest = inv["is_guest"] == 1
inv.loc[guest, ["n_prev_orders", "avg_prev_spend", "days_since_last"]] = [0, np.nan, np.nan]

inv["y_log"] = np.log1p(inv["InvoiceTotal"])
print(inv.shape)
inv.head()

## 5. Trực quan hóa

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(inv["InvoiceTotal"], bins=60, ax=ax[0]); ax[0].set_title("InvoiceTotal (thang gốc, lệch phải)")
sns.histplot(inv["y_log"], bins=60, ax=ax[1], color="C1"); ax[1].set_title("log1p(InvoiceTotal) (gần chuẩn hơn)")
plt.tight_layout(); plt.savefig(OUT_DIR / "01_target_dist.png", dpi=120); plt.show()

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 8))
inv.groupby("Month")["InvoiceTotal"].sum().plot(kind="bar", ax=ax[0, 0], title="Tổng doanh thu theo tháng")
inv.groupby("DayOfWeek")["InvoiceTotal"].mean().plot(kind="bar", ax=ax[0, 1], title="Giá trị hóa đơn TB theo thứ (0=Thứ 2)")
inv.groupby("Hour")["InvoiceTotal"].sum().plot(kind="bar", ax=ax[1, 0], title="Tổng doanh thu theo giờ")
inv.groupby("Country")["InvoiceTotal"].mean().sort_values().plot(kind="barh", ax=ax[1, 1], title="Giá trị hóa đơn TB theo quốc gia")
plt.tight_layout(); plt.savefig(OUT_DIR / "02_eda_time_country.png", dpi=120); plt.show()

In [ ]:
daily = inv.set_index("InvoiceDate")["InvoiceTotal"].resample("W").sum()
daily.plot(figsize=(13, 3.5), title="Doanh thu theo tuần")
plt.axvline(pd.Timestamp(SPLIT_DATE), color="red", ls="--", label="Mốc chia train/test"); plt.legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "03_weekly_sales.png", dpi=120); plt.show()

In [ ]:
num_cols = ["Month", "DayOfMonth", "DayOfWeek", "Hour", "IsWeekend", "n_lines",
            "n_prev_orders", "avg_prev_spend", "days_since_last", "is_guest", "InvoiceTotal"]
plt.figure(figsize=(9, 6))
sns.heatmap(inv[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Ma trận tương quan"); plt.tight_layout(); plt.savefig(OUT_DIR / "04_corr.png", dpi=120); plt.show()

## 6. Chuẩn bị huấn luyện (Chapter 5)

- **Chia theo thời gian** (không chia ngẫu nhiên, tránh lộ thông tin tương lai).
- **Pipeline**: số → impute median (+ cờ missing) → `StandardScaler`; `Country` → `OneHotEncoder`. Pipeline chỉ `fit` trên train.

In [ ]:
num_feats = ["Month", "DayOfMonth", "DayOfWeek", "Hour", "IsWeekend", "n_lines",
             "n_prev_orders", "avg_prev_spend", "days_since_last", "is_guest"]
cat_feats = ["Country"]

train = inv[inv["InvoiceDate"] < SPLIT_DATE]
test = inv[inv["InvoiceDate"] >= SPLIT_DATE]
X_train, y_train_log, y_train = train[num_feats + cat_feats], train["y_log"], train["InvoiceTotal"]
X_test, y_test_log, y_test = test[num_feats + cat_feats], test["y_log"], test["InvoiceTotal"]
print(f"Train: {len(train):,} ({train.InvoiceDate.min():%Y-%m-%d} -> {train.InvoiceDate.max():%Y-%m-%d})")
print(f"Test : {len(test):,} ({test.InvoiceDate.min():%Y-%m-%d} -> {test.InvoiceDate.max():%Y-%m-%d})")

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc", StandardScaler())]), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])

## 7. Huấn luyện & đánh giá (Chapter 6, 7, 9, 10)

Mô hình được huấn luyện trên `log1p(InvoiceTotal)`, dự đoán rồi `expm1` về thang gốc để tính metric:

$$\text{MSE}=\frac1n\sum(y_i-\hat y_i)^2,\quad \text{RMSE}=\sqrt{\text{MSE}},\quad \text{MAE}=\frac1n\sum|y_i-\hat y_i|,\quad R^2=1-\frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$$

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)

def evaluate(name, model, max_train=None):
    X_tr, y_tr = X_train, y_train_log
    if max_train and len(X_tr) > max_train:
        idx = np.sort(rng.choice(len(X_tr), max_train, replace=False))
        X_tr, y_tr = X_tr.iloc[idx], y_tr.iloc[idx]
    pipe = Pipeline([("prep", clone(preprocess)), ("model", model)])
    t0 = time.time(); pipe.fit(X_tr, y_tr); fit_t = time.time() - t0
    t0 = time.time(); pred_log = pipe.predict(X_test); pred_t = time.time() - t0
    pred = np.clip(np.expm1(pred_log), 0, None)
    mse = mean_squared_error(y_test, pred)
    res = {"Model": name, "n_train": len(X_tr), "MSE": mse, "RMSE": np.sqrt(mse),
           "MAE": mean_absolute_error(y_test, pred), "R2": r2_score(y_test, pred),
           "R2_log": r2_score(y_test_log, pred_log), "Fit(s)": round(fit_t, 2), "Predict(s)": round(pred_t, 2)}
    print(f"{name:<22} R2={res['R2']:.3f}  R2_log={res['R2_log']:.3f}  MAE={res['MAE']:.1f}  fit={fit_t:.1f}s")
    return pipe, pred, res

models = {
    "Baseline (mean)":   (DummyRegressor(strategy="mean"), None),
    "Linear Regression": (LinearRegression(), None),
    "Ridge":             (Ridge(alpha=1.0), None),
    "Lasso":             (Lasso(alpha=0.001), None),
    "Decision Tree":     (DecisionTreeRegressor(max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE), None),
    "Random Forest":     (RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE), None),
    "KNN":               (KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=-1), MAX_TRAIN_KNN),
    "SVR (RBF)":         (SVR(C=3.0, epsilon=0.1), MAX_TRAIN_SVR),
}
fitted, preds, rows = {}, {}, []
for name, (m, cap_n) in models.items():
    fitted[name], preds[name], r = evaluate(name, m, cap_n)
    rows.append(r)
results = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
results.to_csv(OUT_DIR / "model_comparison.csv", index=False)
results.round(3)

> **Cách đọc:** `R2` (thang gốc) thường thấp hơn `R2_log` vì vài hóa đơn lớn chi phối sai số. Baseline có `R2 ≈ 0` hoặc âm nhẹ; mọi mô hình tốt hơn baseline mới thực sự học được tín hiệu. `n_lines` là đặc trưng mạnh nhất, vì hóa đơn nhiều dòng hàng thì giá trị thường lớn.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
order = results["Model"]
sns.barplot(data=results, y="Model", x="R2", order=order, ax=ax[0]); ax[0].set_title("R² (thang gốc) - cao hơn tốt hơn")
sns.barplot(data=results, y="Model", x="MAE", order=order, ax=ax[1]); ax[1].set_title("MAE - thấp hơn tốt hơn")
sns.barplot(data=results, y="Model", x="Fit(s)", order=order, ax=ax[2]); ax[2].set_title("Thời gian huấn luyện (s)")
plt.tight_layout(); plt.savefig(OUT_DIR / "05_model_comparison.png", dpi=120); plt.show()

## 8. Tinh chỉnh siêu tham số (`TimeSeriesSplit`)

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

# --- Random Forest ---
rf_pipe = Pipeline([("prep", clone(preprocess)),
                    ("model", RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE))])
rf_search = RandomizedSearchCV(
    rf_pipe,
    {"model__n_estimators": [100, 200, 300],
     "model__max_depth": [8, 12, 16, None],
     "model__min_samples_leaf": [3, 5, 10, 20],
     "model__max_features": [0.5, 0.8, 1.0]},
    n_iter=10, cv=tscv, scoring="neg_mean_absolute_error", random_state=RANDOM_STATE, n_jobs=1, verbose=1)
rf_search.fit(X_train, y_train_log)
print("RF best params:", rf_search.best_params_)

# --- SVR (trên mẫu nhỏ để tuning nhanh) ---
sub = np.sort(rng.choice(len(X_train), min(6000, len(X_train)), replace=False))
svr_pipe = Pipeline([("prep", clone(preprocess)), ("model", SVR())])
svr_search = GridSearchCV(svr_pipe, {"model__C": [1, 3, 10], "model__epsilon": [0.05, 0.1, 0.3]},
                          cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1)
svr_search.fit(X_train.iloc[sub], y_train_log.iloc[sub])
print("SVR best params:", svr_search.best_params_)

In [ ]:
rf_best = rf_search.best_estimator_["model"]
svr_best = svr_search.best_estimator_["model"]
extra = []
for name, m, cap_n in [("Random Forest (tuned)", clone(rf_best), None),
                       ("SVR (tuned)", clone(svr_best), MAX_TRAIN_SVR)]:
    fitted[name], preds[name], r = evaluate(name, m, cap_n)
    extra.append(r)
results = (pd.concat([results, pd.DataFrame(extra)], ignore_index=True)
             .sort_values("R2", ascending=False).reset_index(drop=True))
results.to_csv(OUT_DIR / "model_comparison.csv", index=False)
results.round(3)

## 9. Diễn giải mô hình

In [ ]:
best_name = results.iloc[0]["Model"]
print("Mô hình tốt nhất theo R²:", best_name)

# Dự đoán vs thực tế
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
p = preds[best_name]
ax[0].scatter(y_test, p, s=6, alpha=0.3)
lim = [0, max(y_test.max(), p.max())]
ax[0].plot(lim, lim, "r--"); ax[0].set_xlabel("Thực tế"); ax[0].set_ylabel("Dự đoán")
ax[0].set_title(f"{best_name}: thực tế vs dự đoán")
sns.histplot(y_test - p, bins=60, ax=ax[1]); ax[1].set_title("Phân phối sai số (thực tế − dự đoán)")
plt.tight_layout(); plt.savefig(OUT_DIR / "06_pred_vs_actual.png", dpi=120); plt.show()

In [ ]:
# Feature importance của Random Forest (tuned)
pipe_rf = fitted["Random Forest (tuned)"]
names = pipe_rf["prep"].get_feature_names_out()
imp = pd.Series(pipe_rf["model"].feature_importances_, index=names).sort_values(ascending=False).head(15)
imp.iloc[::-1].plot(kind="barh", figsize=(8, 5), title="Top 15 đặc trưng quan trọng (Random Forest)")
plt.tight_layout(); plt.savefig(OUT_DIR / "07_feature_importance.png", dpi=120); plt.show()

In [ ]:
# Cây quyết định nông để dễ đọc quy luật
tree_pipe = Pipeline([("prep", clone(preprocess)),
                      ("model", DecisionTreeRegressor(max_depth=3, min_samples_leaf=50, random_state=RANDOM_STATE))])
tree_pipe.fit(X_train, y_train_log)
plt.figure(figsize=(18, 7))
plot_tree(tree_pipe["model"], feature_names=tree_pipe["prep"].get_feature_names_out(),
          filled=True, rounded=True, fontsize=8)
plt.title("Decision Tree (max_depth=3) trên log1p(InvoiceTotal)")
plt.savefig(OUT_DIR / "08_tree.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# Lỗi theo tháng trong tập test (kiểm tra độ ổn định theo thời gian)
tmp = test[["InvoiceDate", "InvoiceTotal"]].copy()
tmp["pred"] = preds[best_name]
tmp["month"] = tmp["InvoiceDate"].dt.to_period("M")
tmp.groupby("month").apply(lambda x: pd.Series({
    "n": len(x), "MAE": mean_absolute_error(x.InvoiceTotal, x.pred),
    "thực tế TB": x.InvoiceTotal.mean(), "dự đoán TB": x.pred.mean()})).round(1)

## 10. Kết luận (điền sau khi chạy)

- Mô hình tốt nhất: xem `outputs/model_comparison.csv` (thường là Random Forest/ensemble cây).
- So với baseline, mô hình giảm MAE bao nhiêu % và `R2` đạt bao nhiêu.
- Đặc trưng quan trọng nhất (xem biểu đồ feature importance): thường là `n_lines`, lịch sử chi tiêu khách, rồi giờ/tháng.
- Hạn chế: không dùng `Quantity`/`Price` nên R² không thể gần 1; test là Q4/2011 (mùa cao điểm) nên có shift phân phối so với train.
- Hướng mở rộng: dự báo doanh thu theo ngày/tuần (chuỗi thời gian), thêm đặc trưng danh mục sản phẩm, thử Gradient Boosting.